<a href="https://colab.research.google.com/github/A-Kuo/Language-Model-Hallucination-Detection-via-Entropy-Divergence/blob/main/colab/gpu_benchmark.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# AED Full GPU Benchmark
## Attention Entropy Divergence — Hallucination Detection

**Runtime:** ~15 min on T4 GPU (free Colab tier)  
**Output:** `results/benchmark_results.json` — auto-commits to repo with GitHub token

### What this does
1. Loads **Pythia-160m** (the model used in the paper)
2. Loads **HaluEval QA** (the benchmark used in the paper)
3. Extracts per-head Shannon entropy and cross-layer KL divergence
4. Trains BiLSTM and logistic regression classifiers
5. Reports AUROC, FPR@90%TPR, F1, and latency
6. **Auto-commits results back to repo** (with your GH_TOKEN)

### Before you run
**Runtime → Change runtime type → T4 GPU**

### Setup GH_TOKEN (for auto-commit)
1. Go to https://github.com/settings/tokens → Generate new token (classic)
2. Check scopes: `repo` (full control of private repositories)
3. Copy token → Secrets tab (🔑) on left sidebar → Add `GH_TOKEN`
4. Token allows the notebook to push results back to the repo

In [ ]:
# ── Load GitHub Token from Secrets ─────────────────────────────────────────
from google.colab import userdata
import os

try:
    GH_TOKEN = userdata.get('GH_TOKEN')
    os.environ['GH_TOKEN'] = GH_TOKEN
    print('✓ GH_TOKEN loaded from Colab Secrets')
    print('  Auto-commit to repo: ENABLED')
except Exception as e:
    GH_TOKEN = None
    print('⚠️  GH_TOKEN not found in Secrets')
    print('  Auto-commit: DISABLED (manual download only)')
    print('  To enable: Secrets tab → Add GH_TOKEN with repo scope')

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode != 0:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
else:
    print('✓ GPU detected:', result.stdout.strip())

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
%pip install -q transformers datasets accelerate scikit-learn
print('✓ Dependencies installed')

In [ ]:
import os, sys
REPO = 'Language-Model-Hallucination-Detection-via-Entropy-Divergence'
if not os.path.exists(REPO):
    !git clone https://github.com/A-Kuo/{REPO}.git
else:
    !git -C {REPO} pull
os.chdir(REPO)
sys.path.insert(0, 'v1')
print('✓ Repo ready:', os.getcwd())

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────
import os
import json
import time
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.makedirs('results', exist_ok=True)

# Import from local v1 module
from run_experiment import (
    load_halueval_qa, extract_features, BiLSTMClassifier,
    auroc, f1_score, fpr_at_tpr
)

print('✓ Imports successful')

In [ ]:
# ── Load Model ────────────────────────────────────────────────────────────
print('Loading Pythia-160m...')
tokenizer = AutoTokenizer.from_pretrained('EleutherAI/pythia-160m')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    'EleutherAI/pythia-160m',
    output_attentions=True,
    torch_dtype=torch.float32
).cuda()
model.eval()

num_layers = model.config.num_hidden_layers
num_heads = model.config.num_attention_heads
print(f'✓ Model loaded: {num_layers} layers, {num_heads} heads')

In [ ]:
# ── Load Dataset ──────────────────────────────────────────────────────────
MAX_SAMPLES = 500
print(f'Loading HaluEval QA (max {MAX_SAMPLES} samples)...')
pairs = load_halueval_qa(max_samples=MAX_SAMPLES)
print(f'✓ Loaded {len(pairs)} samples')

halluc_count = sum(1 for _, _, label in pairs if label == 1)
correct_count = len(pairs) - halluc_count
print(f'  Hallucinated: {halluc_count}, Correct: {correct_count}')

In [ ]:
# ── Extract Features ─────────────────────────────────────────────────────
print('\nExtracting attention features...')
features = []
labels = []
latencies = []

for i, (ctx, cont, label) in enumerate(pairs):
    t0 = time.perf_counter()
    feat = extract_features(model, tokenizer, ctx, cont, 'cuda')
    t1 = time.perf_counter()
    
    # Skip NaN (zero attention weights cause log(0) = -inf)
    if np.any(np.isnan(feat)):
        print(f'  Warning: NaN in sample {i}, skipping')
        continue
    
    features.append(feat)
    labels.append(label)
    latencies.append(t1 - t0)
    
    if (i + 1) % 50 == 0:
        avg_lat = np.mean(latencies[-50:]) * 1000
        print(f'  {i+1}/{len(pairs)} done ({avg_lat:.0f}ms/sample)')

X = np.array(features)
y = np.array(labels)
print(f'\n✓ Feature extraction: {X.shape} (skipped {len(pairs) - len(features)} NaN samples)')
print(f'  Mean latency: {np.mean(latencies)*1000:.1f} ms/sample')

In [ ]:
# ── Train and Evaluate ──────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train)}, Test: {len(X_test)}')

# Train BiLSTM
print('\nTraining BiLSTM classifier...')
bilstm = BiLSTMClassifier(input_dim=X.shape[1])
bilstm.fit(X_train, y_train.astype(np.float32))
bilstm_proba = bilstm.predict_proba(X_test)

# Train LogReg baseline
print('Training Logistic Regression baseline...')
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)
logreg_proba = logreg.predict_proba(X_test)[:, 1]

# Compute metrics
results = {
    'model_name': 'EleutherAI/pythia-160m',
    'num_samples': len(y),
    'num_layers': num_layers,
    'device': 'cuda',
    'aed_auroc': round(auroc(y_test, bilstm_proba), 4),
    'aed_f1': round(f1_score(y_test, (bilstm_proba > 0.5).astype(int)), 4),
    'aed_fpr90': round(fpr_at_tpr(y_test, bilstm_proba), 4),
    'aed_latency_ms': round(float(np.mean(latencies) * 1000), 2),
    'logreg_auroc': round(auroc(y_test, logreg_proba), 4),
    'logreg_f1': round(f1_score(y_test, (logreg_proba > 0.5).astype(int)), 4),
    'logreg_fpr90': round(fpr_at_tpr(y_test, logreg_proba), 4),
}

print('\n' + '='*60)
print('  FINAL RESULTS')
print('='*60)
for k, v in results.items():
    print(f'  {k:20}: {v}')
print('='*60)

In [ ]:
# ── Save Results ──────────────────────────────────────────────────────────
import json

with open('results/benchmark_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('✓ Saved: results/benchmark_results.json')

# Display LaTeX placeholders (using format() to avoid backslash issues)
print('\n' + '='*60)
print('  LATEX PLACEHOLDERS — Copy to paper.tex')
print('='*60)
print('\RESULT{{{:.3f}}}  ← AED BiLSTM AUROC'.format(results['aed_auroc']))
print('\RESULT{{{:.3f}}}  ← LogReg Baseline AUROC'.format(results['logreg_auroc']))
print('{:.1f} ms    ← Latency per sample'.format(results['aed_latency_ms']))
print('='*60)

In [ ]:
# ── Generate Visualization ─────────────────────────────────────────────
import matplotlib.pyplot as plt

print('Generating per-layer entropy visualization...')
viz_pairs = pairs[:40]

correct_entropy = np.zeros(num_layers)
halluc_entropy = np.zeros(num_layers)
correct_count = halluc_count = 0

for ctx, cont, label in viz_pairs:
    feats = extract_features(model, tokenizer, ctx, cont, 'cuda')
    if np.any(np.isnan(feats)):
        continue
    if label == 0:
        correct_entropy += feats[:num_layers]
        correct_count += 1
    else:
        halluc_entropy += feats[:num_layers]
        halluc_count += 1

if correct_count > 0 and halluc_count > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    layers = range(1, num_layers + 1)
    ax.plot(layers, correct_entropy/correct_count, 'b-o', label='Non-hallucinated')
    ax.plot(layers, halluc_entropy/halluc_count, 'r-o', label='Hallucinated')
    ax.set_xlabel('Layer')
    ax.set_ylabel('Mean Attention Entropy')
    ax.set_title('Per-Layer Attention Entropy: Correct vs Hallucinated')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.savefig('results/figure_layer_entropy.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('✓ Saved: results/figure_layer_entropy.png')
else:
    print('⚠️  Not enough valid samples for visualization')

In [ ]:
# ── Auto-commit results back to GitHub ───────────────────────────────────
import os

GH_TOKEN = os.environ.get('GH_TOKEN')

if not GH_TOKEN:
    print('⚠️  No GH_TOKEN — downloading manually')
    from google.colab import files
    for f in ['results/benchmark_results.json', 'results/figure_layer_entropy.png']:
        if os.path.exists(f):
            files.download(f)
            print(f'  Downloaded: {f}')
elif os.path.exists('results/benchmark_results.json'):
    with open('results/benchmark_results.json') as f:
        r = json.load(f)
    auroc_val = r.get('aed_auroc', 0.0)
    
    remote_url = f'https://{GH_TOKEN}@github.com/A-Kuo/Language-Model-Hallucination-Detection-via-Entropy-Divergence.git'
    !git remote set-url origin {remote_url}
    !git config user.email "colab@gpu-benchmark.ai"
    !git config user.name "GPU Benchmark Bot"
    !git add results/
    !git commit -m "results: GPU benchmark (AUROC={auroc_val:.3f})" || echo 'Nothing new'
    !git push origin main
    print('✓ Results committed to repo!')
else:
    print('⚠️  No results to commit')